In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [ ]:
!pip uninstall -y transformers -q

!pip install -q --no-cache-dir git+https://github.com/huggingface/transformers.git

!pip install -q -U qwen-asr

!pip install -q -U accelerate peft bitsandbytes datasets evaluate jiwer librosa soundfile

!pip install -q "torchao>=0.16.0"

# Imports

In [ ]:
import os
import gc
import glob
import torch
import zipfile
import evaluate
import pandas as pd

from qwen_asr import Qwen3ASRModel
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor

from datasets import Dataset, Audio
from huggingface_hub import hf_hub_download, login

from transformers import (
    BitsAndBytesConfig,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    TrainerCallback,
    AutoModelForSpeechSeq2Seq
)

from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
    PeftModel
)

from dataclasses import dataclass
from typing import Any, Dict, List, Union

print("Torch:", torch.__version__)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# Config

In [ ]:
MODEL_NAME = "Qwen/Qwen3-ASR-0.6B"

BASE_DIR = "/kaggle/working/khotbah_dataset"

OUTPUT_DIR = "/kaggle/working/qwen3_asr_lora"

MAX_AUDIO_LENGTH = 30

BATCH_SIZE = 16
GRAD_ACCUM = 8

LEARNING_RATE = 2e-5
NUM_EPOCHS = 3

TOTAL_PARTS = 2

torch.backends.cuda.matmul.allow_tf32 = True

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
login("")

# Dataset

In [ ]:
# ============================================================
# DOWNLOAD DATASET
# ============================================================

os.makedirs(BASE_DIR, exist_ok=True)

REPO_ID = "gracecalista/new-dataset-transcribe"

print("Mendownload metadata...")

csv_path = hf_hub_download(
    repo_id=REPO_ID,
    filename="benchmarking_results_final.csv",
    repo_type="dataset",
    local_dir=BASE_DIR
)

df = pd.read_csv(csv_path).dropna(subset=['text', 'path'])

for part_num in range(1, TOTAL_PARTS + 1):

    EXTRACT_FLAG = os.path.join(
        BASE_DIR,
        f".extracted_part{part_num}"
    )

    zip_filename = f"wavs/wavs_part{part_num}.zip"

    if os.path.exists(EXTRACT_FLAG):

        with open(EXTRACT_FLAG, 'r') as f:
            flag_content = f.read()

        print(f"Part {part_num} sudah ada ({flag_content})")
        continue

    print(f"\n Downloading {zip_filename}...")

    zip_path = hf_hub_download(
        repo_id=REPO_ID,
        filename=zip_filename,
        repo_type="dataset",
        local_dir=BASE_DIR
    )

    print("Extracting...")

    with zipfile.ZipFile(zip_path, 'r') as zip_ref:

        members = zip_ref.infolist()
        total = len(members)

        for i, member in enumerate(members, 1):

            zip_ref.extract(member, BASE_DIR)

            if i % 5000 == 0 or i == total:
                print(f"{i}/{total}")

    extracted_wavs = glob.glob(
        f"{BASE_DIR}/**/*.wav",
        recursive=True
    )

    with open(EXTRACT_FLAG, 'w') as f:
        f.write(f"done={len(extracted_wavs)}")

    os.remove(zip_path)

    print(f"Part {part_num} selesai")

# ============================================================
# BUILD DATASET
# ============================================================

all_wavs = glob.glob(
    f"{BASE_DIR}/**/*.wav",
    recursive=True
)

print("TOTAL WAV:", len(all_wavs))

path_map = {
    os.path.basename(p): p
    for p in all_wavs
}

df['audio_path'] = df['path'].apply(
    lambda x: path_map.get(os.path.basename(x))
)

df = df.dropna(subset=['audio_path'])

print("VALID DATA:", len(df))

df = df.head(45000)

ds = Dataset.from_dict({
    "audio": df["audio_path"].tolist(),
    "sentence": df["text"].tolist()
})

ds = ds.cast_column(
    "audio",
    Audio(sampling_rate=32000, decode=True)
)

ds = ds.train_test_split(
    test_size=0.1,
    seed=42
)

print(ds)

# Filter Audio Corrupt

In [ ]:
def is_valid_audio(example):

    try:
        _ = example["audio"]["array"]
        return True

    except:
        return False

ds = ds.filter(is_valid_audio)

print(ds)

# Load Processor

In [ ]:
from qwen_asr.core.transformers_backend.processing_qwen3_asr import Qwen3ASRProcessor

_processor = Qwen3ASRProcessor.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)

_processor.tokenizer.padding_side = "left"

print(type(_processor))

# BNB Config

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

In [ ]:
from qwen_asr import Qwen3ASRModel

model = Qwen3ASRModel.from_pretrained(
    MODEL_NAME,
    dtype=torch.bfloat16,
    device_map="cuda:0",
)

inner_model = model.model  
inner_model.thinker.config.pad_token_id = _processor.tokenizer.pad_token_id
inner_model.thinker.generation_config.pad_token_id = _processor.tokenizer.pad_token_id

print(model.__class__.__name__)
print(inner_model.__class__.__name__)

In [ ]:
from peft import PeftModel

thinker = inner_model.thinker
thinker.config.use_cache = False

thinker = prepare_model_for_kbit_training(
    thinker,
    use_gradient_checkpointing=True
)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ],
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM",
)

thinker = get_peft_model(thinker, lora_config)
thinker.print_trainable_parameters()

inner_model.thinker = thinker

# Preprocess Function

In [ ]:
import numpy as np
from datasets import Dataset, Audio

import numpy as np
import librosa

def prepare_batch(batch, processor):

    input_features_list = []
    labels_list = []

    for audio, sentence in zip(batch["audio"], batch["sentence"]):

        # Handle torchcodec AudioDecoder
        if hasattr(audio["array"], "numpy"):
            audio_array = audio["array"].numpy()
        else:
            audio_array = np.array(audio["array"])

        # Flatten kalau stereo
        if audio_array.ndim > 1:
            audio_array = audio_array.mean(axis=0)

        audio_array = audio_array.astype(np.float32)
        sampling_rate = audio["sampling_rate"]

        # Resample ke 16000 kalau perlu
        if sampling_rate != 16000:
            audio_array = librosa.resample(
                audio_array,
                orig_sr=sampling_rate,
                target_sr=16000
            )
            sampling_rate = 16000

        max_length = sampling_rate * MAX_AUDIO_LENGTH

        if len(audio_array) > max_length:
            audio_array = audio_array[:max_length]

        features = processor.feature_extractor(
            audio_array,
            sampling_rate=16000,
            return_tensors="np"
        ).input_features[0]

        labels = processor.tokenizer(
            sentence,
            return_tensors="np"
        ).input_ids[0]

        input_features_list.append(features)
        labels_list.append(labels)

    return {
        "input_features": input_features_list,
        "labels": labels_list,
    }

# Map Dataset
> hanya normalize teks 

In [ ]:
import re

def normalize_text(example):
    text = example["sentence"].lower()
    text = re.sub(r"\s+", " ", text).strip()
    return {"sentence": text}

ds = ds.map(normalize_text, num_proc=2)

# Cast audio tapi TIDAK diproses di sini
ds = ds.cast_column("audio", Audio(sampling_rate=16000))

print(ds)

In [ ]:
import torch
import torch.nn as nn
from transformers.modeling_outputs import Seq2SeqLMOutput

class Qwen3ASRTrainableModel(nn.Module):

    def __init__(self, qwen_asr_model):
        super().__init__()
        self.inner = qwen_asr_model  # Qwen3ASRForConditionalGeneration
        self.config = qwen_asr_model.config

    def forward(
        self,
        input_features=None,
        labels=None,
        **kwargs
    ):
        # Forward lewat full model
        outputs = self.inner(
            input_features=input_features,
            labels=labels,
            **kwargs
        )
        return outputs

    def generate(self, input_features=None, **kwargs):
        return self.inner.generate(
            input_features=input_features,
            **kwargs
        )

    def gradient_checkpointing_enable(self, **kwargs):
        if hasattr(self.inner, "gradient_checkpointing_enable"):
            self.inner.gradient_checkpointing_enable(**kwargs)

    def parameters(self, recurse=True):
        return self.inner.parameters(recurse=recurse)

    def named_parameters(self, **kwargs):
        return self.inner.named_parameters(**kwargs)


# Wrap model
trainable_model = Qwen3ASRTrainableModel(inner_model)

# Data Collator

In [ ]:
def left_pad_sequence(sequences, padding_value):
    max_len = max(s.shape[0] for s in sequences)
    padded = []
    for s in sequences:
        pad_len = max_len - s.shape[0]
        padded.append(torch.cat([
            torch.full((pad_len,), padding_value, dtype=s.dtype),
            s
        ]))
    return torch.stack(padded)

@dataclass
class DataCollatorSpeechSeq2Seq:

    processor: Any

    def __call__(
        self,
        features: List[Dict[str, Union[List[int], torch.Tensor]]]
    ) -> Dict[str, torch.Tensor]:

        all_input_ids = []
        all_attention_mask = []
        all_feature_attention_mask = []
        all_input_features = []
        all_labels = []

        eos_id = self.processor.tokenizer.eos_token_id  # 151645 = <|im_end|>

        for f in features:

            audio_array = np.array(f["audio"]["array"]).astype(np.float32)
            sampling_rate = f["audio"]["sampling_rate"]

            if sampling_rate != 16000:
                audio_array = librosa.resample(
                    audio_array,
                    orig_sr=sampling_rate,
                    target_sr=16000
                )

            max_length = 16000 * MAX_AUDIO_LENGTH
            if len(audio_array) > max_length:
                audio_array = audio_array[:max_length]

            processed = self.processor(
                audio=audio_array,
                sampling_rate=16000,
                text=f["sentence"],
                return_tensors="pt"
            )

            # Append EOS ke input_ids supaya model belajar kapan stop
            input_ids = processed["input_ids"][0]
            if input_ids[-1] != eos_id:
                input_ids = torch.cat([input_ids, torch.tensor([eos_id])])

            # Append 1 ke attention_mask supaya EOS di-attend
            attention_mask = processed["attention_mask"][0]
            attention_mask = torch.cat([attention_mask, torch.tensor([1])])

            all_input_ids.append(input_ids)
            all_attention_mask.append(attention_mask)
            all_feature_attention_mask.append(processed["feature_attention_mask"][0])
            all_input_features.append(processed["input_features"][0])

            # Label = input_ids (sudah include EOS), pad token di-mask -100
            label_ids = input_ids.clone()
            all_labels.append(label_ids)

        # Left pad input_ids
        input_ids_padded = left_pad_sequence(
            all_input_ids,
            padding_value=self.processor.tokenizer.pad_token_id
        )

        # Left pad attention_mask
        attention_mask_padded = left_pad_sequence(
            all_attention_mask,
            padding_value=0
        )

        # Left pad labels dengan -100 (pad token tidak dihitung dalam loss)
        labels_padded = left_pad_sequence(
            all_labels,
            padding_value=-100
        )

        # Pad input_features dan feature_attention_mask ke max length
        max_feat_len = max(f.shape[-1] for f in all_input_features)

        padded_features = []
        padded_feat_mask = []

        for feat, mask in zip(all_input_features, all_feature_attention_mask):
            pad_len = max_feat_len - feat.shape[-1]
            padded_features.append(torch.nn.functional.pad(feat, (0, pad_len), value=0.0))
            padded_feat_mask.append(torch.nn.functional.pad(mask, (0, pad_len), value=0))

        input_features_stacked = torch.stack(padded_features)
        feature_attention_mask_stacked = torch.stack(padded_feat_mask)

        return {
            "input_ids": input_ids_padded,
            "attention_mask": attention_mask_padded,
            "input_features": input_features_stacked,
            "feature_attention_mask": feature_attention_mask_stacked,
            "labels": labels_padded,
        }

data_collator = DataCollatorSpeechSeq2Seq(processor=_processor)

# Metrics - WER

In [ ]:
wer_metric = evaluate.load("wer")

import re

def compute_metrics(pred):

    pred_ids = pred.predictions
    label_ids = pred.label_ids.copy()

    # Clip supaya tidak overflow
    vocab_size = _processor.tokenizer.vocab_size
    pred_ids = np.clip(pred_ids, 0, vocab_size - 1).astype(np.int32)

    label_ids[label_ids == -100] = _processor.tokenizer.pad_token_id

    pred_str = _processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = _processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    def clean_pred(text):
        text = text.replace('⽗', '')        # karakter dari token id out-of-range
        text = re.sub(r'language-\w+', '', text)
        text = re.sub(r'<[^>]+>', '', text)
        text = text.split('\n')[0]
        text = text.strip().lower()
        return text

    pred_str = [clean_pred(p) for p in pred_str]
    label_str = [t.strip().lower() for t in label_str]

    print("\n===== SAMPLE PREDICTIONS =====")
    for i in range(min(3, len(pred_str))):
        print(f"[{i+1}] PRED : {pred_str[i]}")
        print(f"      LABEL: {label_str[i]}")
        print()

    pred_str = [p if p.strip() else "[empty]" for p in pred_str]

    wer = wer_metric.compute(predictions=pred_str, references=label_str)
    return {"wer": wer}

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    num_train_epochs=NUM_EPOCHS,
    fp16=True,   
    bf16=False,
    logging_steps=10,
    eval_steps=100,
    save_steps=100,
    eval_strategy="steps",
    save_strategy="steps",
    predict_with_generate=True,
    generation_max_length=225,
    warmup_steps=100,
    lr_scheduler_type="cosine",
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    report_to="none",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    save_total_limit=3,
    remove_unused_columns=False,
    dataloader_num_workers=0,
    optim="paged_adamw_8bit",
    push_to_hub=True,
    hub_model_id="gracecalista/qwen3-asr-khotbah-lora-final",
    hub_strategy="checkpoint",
)

In [ ]:
from transformers import TrainerCallback
from huggingface_hub import HfApi, ModelCard, ModelCardData

class ForcePushCheckpointCallback(TrainerCallback):
    def on_save(self, args, state, control, **kwargs):
        if state.is_world_process_zero:
            api = HfApi()
            checkpoint_folder = f"{args.output_dir}/checkpoint-{state.global_step}"

            # Buat README.md dengan base_model yang benar
            card = ModelCard.from_template(
                ModelCardData(
                    base_model="Qwen/Qwen3-ASR-0.6B",
                    language=["id"],
                    tags=["speech", "asr", "lora", "indonesian"],
                ),
                model_id="gracecalista/qwen3-asr-khotbah-lora-final",
            )
            card.save(f"{checkpoint_folder}/README.md")

            api.upload_folder(
                folder_path=checkpoint_folder,
                repo_id="gracecalista/qwen3-asr-khotbah-lora",
                path_in_repo=f"checkpoint-{state.global_step}",
                repo_type="model",
            )
            print(f"✅ Pushed checkpoint-{state.global_step} to HF Hub")

In [ ]:
from transformers import EarlyStoppingCallback

trainer = Seq2SeqTrainer(
    model=inner_model.thinker,
    args=training_args,
    train_dataset=ds["train"],
    eval_dataset=ds["test"],
    data_collator=data_collator,
    processing_class=_processor,
    compute_metrics=compute_metrics,
    callbacks=[
        EarlyStoppingCallback(early_stopping_patience=3), 
        ForcePushCheckpointCallback()  
    ],
)

In [ ]:
gc.collect()
torch.cuda.empty_cache()

inner_model.thinker.enable_input_require_grads()

trainer.train() # trainer.train(resume_from_checkpoint="/kaggle/working/checkpoint-400") --> menyesuaikan checkpoint terakhir